In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import seaborn as sns
import matplotlib.pyplot as plt
import hdf5plugin
import h5py
import numpy as np
import pandas as pd

In [ ]:
# --- config ---
in_dir = "../../results/03_accessbpnet/14_marginal_footprint"
out_dir = "../../results/03_accessbpnet/20_viz_marginal_footprint"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
path = f"{in_dir}/K562/ATAC/uncorrected/_footprints.h5"

In [ ]:
samples = ["K562", "HepG2"]
assays = ["ACCESS", "ATAC"]
modes = ["corrected", "uncorrected"]
flank = 100
assay_color = {"ACCESS": "#5e3c99", "ATAC": "#1b7837"}
mode_style = {"corrected": "-", "uncorrected": "*"}

PROFILE_KEY = "i0"   # <-- adjust after peeking (deepdish list encoding)


def load_footprints(sample, assay, mode):
    """Load one footprints.h5 -> dict{motif: profile_array} using h5py."""
    path = f"{in_dir}/{sample}/{assay}/{mode}/_footprints.h5"
    if not os.path.exists(path):
        print(f"  [missing] {path}")
        return {}
    out = {}
    with h5py.File(path, "r") as f:
        for motif in f.keys():
            grp = f[motif]
            if isinstance(grp, h5py.Group):
                # list stored as i0 (profile), i1 (counts)
                out[motif] = grp[PROFILE_KEY][:].astype(np.float32)
            else:
                # stored directly as a dataset
                out[motif] = grp[:].astype(np.float32)
    return out


def center_slice(profile, flank):
    mid = len(profile) // 2
    return profile[mid - flank: mid + flank]


# --- load: data[sample][assay][mode] = {motif: profile} ---
data = {s: {a: {m: load_footprints(s, a, m) for m in modes}
            for a in assays} for s in samples}

In [ ]:
# convert data to dataframe
df_list = []
for s in samples:
    for a in assays:
        for m in modes:
            for motif, profile in data[s][a][m].items():
                for i, val in enumerate(center_slice(profile, flank)):
                    df_list.append({
                        "sample": s,
                        "assay": a,
                        "mode": m,
                        "motif": motif,
                        "position": i - flank,
                        "value": val
                    })
df_data = pd.DataFrame(df_list)
df_data.to_csv(f"{out_dir}/marginal_footprints.csv", index=False)

In [ ]:
# --- collect motifs ---
all_motifs = set()
for s in samples:
    for a in assays:
        for m in modes:
            all_motifs.update(data[s][a][m].keys())
all_motifs = sorted(x for x in all_motifs if x != "control")
print(f"{len(all_motifs)} motifs found")

# --- one figure per motif; rows=samples, cols=assays (corrected and uncorrected)
for motif in all_motifs:
    fig, axes = plt.subplots(len(samples), len(assays)*2,
                             figsize=(5 * len(assays) * 2, 3.5 * len(samples)),
                             squeeze=False)
    for r, sample in enumerate(samples):
        for c, assay in enumerate(assays):

            ax = axes[r, c*2]  # corrected
            prof = data[sample][assay]["corrected"].get(motif)
            if prof is not None:
                y = center_slice(prof, flank)
                x = np.arange(-len(y) // 2, len(y) // 2)
                ax.plot(x, y, color=assay_color[assay],
                        lw=1.2, label="corrected")

                ax.set_title(f"{sample} — {assay}", fontsize=10)
                ax.set_xlabel("Distance to motif center (bp)")
                ax.set_ylabel("Footprint score")
                ax.axvline(0, color="grey", lw=0.5, linestyle=":")
                ax.legend(fontsize=8, frameon=False)

            ax = axes[r, c*2+1]  # uncorrected
            prof = data[sample][assay]["uncorrected"].get(motif)
            if prof is not None:
                y = center_slice(prof, flank)
                x = np.arange(-len(y) // 2, len(y) // 2)
                ax.plot(x, y, color=assay_color[assay],
                        lw=1.2, label="uncorrected")

                ax.set_title(f"{sample} — {assay}", fontsize=10)
                ax.set_xlabel("Distance to motif center (bp)")
                ax.set_ylabel("Footprint score")
                ax.axvline(0, color="grey", lw=0.5, linestyle=":")
                ax.legend(fontsize=8, frameon=False)

    
    fig.suptitle(motif, fontsize=12, y=1.0)
    fig.tight_layout()
    safe = motif.replace("::", "_").replace("/", "_")
    fig.savefig(f"{out_dir}/{safe}.png",
                dpi=150, bbox_inches="tight")

    plt.close(fig)